# Обработка данных и добавление признаков


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer
import warnings
warnings.filterwarnings('ignore')

## 1. Загрузка данных

In [3]:
df_train = pd.read_parquet('train_main_features.parquet')
df_test  = pd.read_parquet('test_main_features.parquet')
df_target = pd.read_parquet('train_target.parquet')

print(f'Train: {df_train.shape}, Test: {df_test.shape}')
print(f'Target: {df_target.shape}')

Train: (750000, 200), Test: (250000, 200)
Target: (750000, 42)


In [4]:
ID_COL     = 'customer_id'
target_cols = [c for c in df_target.columns if c != ID_COL]
num_cols   = [c for c in df_train.columns if c.startswith('num_feature_')]
cat_cols   = [c for c in df_train.columns if c.startswith('cat_feature_')]

print(f'Числовых: {len(num_cols)}, Категориальных: {len(cat_cols)}, Таргетов: {len(target_cols)}')

Числовых: 132, Категориальных: 67, Таргетов: 41


## 2. `_isnan` 

Для каждого числового признака с хотя бы одним NaN создаём бинарный флаг 
nan меняем на 0 -- для моделей которые не принимают наны

делаем доп колонку, число не нан фичей. Типо показатель активности клиента

In [7]:
# Признаки с хотя бы одним NaN (считаем на train)
null_rates = df_train[num_cols].isnull().mean()
isnan_flag_cols = null_rates[null_rates > 0].index.tolist()

isnan_cols = []
for f in isnan_flag_cols:
    new_col = f'{f}_isnan'
    df_train[new_col] = df_train[f].isnull().astype(np.int8)
    df_test[new_col]  = df_test[f].isnull().astype(np.int8)
    isnan_cols.append(new_col)


df_train['count_not_null'] = len(isnan_flag_cols) - df_train[isnan_cols].sum(axis=1)
df_test['count_not_null']  = len(isnan_flag_cols) - df_test[isnan_cols].sum(axis=1)

print(f'Числовых признаков с NaN: {len(isnan_flag_cols)}')
print(f'Создано _isnan флагов:     {len(isnan_cols)}')

Числовых признаков с NaN: 132
Создано _isnan флагов:     132


## 5. `is_whale` флаги

отдельно выдеяем лифт фичи


In [8]:
# Топ whale-признаков из EDA (сильнейшие lift-эффекты)
whale_features = [
    'num_feature_10',
    'num_feature_36',
    'num_feature_56',
    'num_feature_62',
    'num_feature_27',
    'num_feature_117',
    'num_feature_112',
]

whale_thresholds = {}
whale_cols = []

for f in whale_features:
    q99 = df_train[f].quantile(0.99)
    whale_thresholds[f] = q99

    new_col = f'is_whale_{f}'
    df_train[new_col] = (df_train[f] >= q99).astype(np.int8)
    df_test[new_col]  = (df_test[f] >= q99).astype(np.int8)
    whale_cols.append(new_col)

print(f'Создано is_whale флагов: {len(whale_cols)}')
print('Доля whale-клиентов в train:')
print(df_train[whale_cols].mean().apply(lambda x: f'{x:.2%}').to_string())

Создано is_whale флагов: 7
Доля whale-клиентов в train:
is_whale_num_feature_10     0.44%
is_whale_num_feature_36     0.89%
is_whale_num_feature_56     0.40%
is_whale_num_feature_62     0.98%
is_whale_num_feature_27     0.96%
is_whale_num_feature_117    0.85%
is_whale_num_feature_112    0.03%


## 5. Попарные взаимодействия категориальных признаков 
Для линейных моделей создаем признаки попарного взаимодействия. Хи2 все прокрашивал, заменил на метрику крамера


In [9]:
from itertools import combinations
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import numpy as np

# Делаем подвыборку, чтобы не ждать слишком долго
SAMPLE_N = 100_000
rng = np.random.default_rng(42)
idx = rng.choice(len(df_train), SAMPLE_N, replace=False)

target_matrix = df_train[[ID_COL]].merge(df_target, on=ID_COL)[target_cols].values
X_s = df_train[cat_cols].astype(int).values[idx]   
Y_s = target_matrix[idx]                            
p_y = Y_s.mean(axis=0)

# V = sqrt(chi / (n * (k-1))), где k — число уникальных значений. Хи квадрат везде прокрашивал значимость
CRAMER_V_THRESHOLD = 0.3

def score_pair(i, j):
    max_j = X_s[:, j].max() + 1
    x = X_s[:, i] * max_j + X_s[:, j]

    unique_vals = np.unique(x)
    k = len(unique_vals)
    if k < 2:
        return None

    scores = np.zeros(Y_s.shape[1])
    for v in unique_vals:
        mask = x == v
        n_v   = mask.sum()
        k_pos = Y_s[mask].sum(axis=0)
        e_pos = n_v * p_y
        e_neg = n_v * (1 - p_y)
        scores += np.where(e_pos > 0, (k_pos - e_pos)**2 / e_pos, 0)
        scores += np.where(e_neg > 0, (n_v - k_pos - e_neg)**2 / e_neg, 0)

    chi2_max = scores.max()
    # V: для бинарного таргета min(k,r)-1 = min(k,2)-1 = 1
    cramer_v = np.sqrt(chi2_max / SAMPLE_N)

    return (cat_cols[i], cat_cols[j], chi2_max, cramer_v) if cramer_v >= CRAMER_V_THRESHOLD else None


all_pairs = list(combinations(range(len(cat_cols)), 2))  # 2211 пар

raw = Parallel(n_jobs=-1, prefer='threads', return_as='generator')(
    delayed(score_pair)(i, j) for i, j in all_pairs
)
pair_scores = sorted(
    [r for r in tqdm(raw, total=len(all_pairs)) if r],
    key=lambda x: -x[3]   # сортируем по  V
)



# Оставляем значимые
interaction_cols = []
for f1, f2, *_ in tqdm(pair_scores):
    col_name = f'cat_ix_{f1}_{f2}'
    max_b = int(df_train[f2].max()) + 1
    df_train[col_name] = (df_train[f1].astype(int) * max_b + df_train[f2].astype(int)).astype(np.int32)
    df_test[col_name]  = (df_test[f1].astype(int) * max_b + df_test[f2].astype(int)).astype(np.int32)
    interaction_cols.append(col_name)

print(f'Создано пар: {len(interaction_cols)}')

  0%|          | 0/2211 [00:00<?, ?it/s]

  0%|          | 0/133 [00:00<?, ?it/s]

Создано пар: 133


## 7. Масштабирование 
масштабируем неприрывные фичи

In [10]:
qt = QuantileTransformer(
    n_quantiles=1000,
    output_distribution='normal',
    random_state=42
)


_train_num = df_train[num_cols].fillna(0)
_test_num  = df_test[num_cols].fillna(0)

qt.fit(_train_num)

qt_col_names = [f'{c}_qt' for c in num_cols]

df_train_qt = pd.DataFrame(
    qt.transform(_train_num),
    columns=qt_col_names,
    index=df_train.index
)
df_test_qt = pd.DataFrame(
    qt.transform(_test_num),
    columns=qt_col_names,
    index=df_test.index
)

print(f'Отмасштабировано признаков: {len(qt_col_names)}')

Отмасштабировано признаков: 132


## 8. One-Hot Encoding категориальных признаков (только для linear)

низко кардинальные просто энкодером
высоко кардинальные делим на бакеты сначала 

In [11]:
from sklearn.preprocessing import OneHotEncoder

COVERAGE_THRESHOLD = 0.85  # 

low_card_cols  = [c for c in cat_cols if df_train[c].nunique() <= 10]
high_card_cols = [c for c in cat_cols if df_train[c].nunique() > 10]


ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe.fit(df_train[low_card_cols])
ohe_col_names = ohe.get_feature_names_out(low_card_cols).tolist()

df_train_ohe = pd.DataFrame(ohe.transform(df_train[low_card_cols]),
                             columns=ohe_col_names, index=df_train.index, dtype=np.int8)
df_test_ohe  = pd.DataFrame(ohe.transform(df_test[low_card_cols]),
                             columns=ohe_col_names, index=df_test.index,  dtype=np.int8)


bucket_encoders = {}   

for f in high_card_cols:
    freq = df_train[f].value_counts(normalize=True)
    cum  = freq.cumsum()
    k    = int((cum < COVERAGE_THRESHOLD).sum()) + 1   # минимум K, покрывающий 90%
    top_k_vals = freq.head(k).index.tolist()
    bucket_encoders[f] = top_k_vals

    print(f'{f}: {df_train[f].nunique()} значений → топ-{k} покрывают '
          f'{freq.head(k).sum():.1%} строк')

    # Заменяем редкие категории на "other"
    bucketed_col = f'{f}_bucket'
    df_train[bucketed_col] = df_train[f].where(df_train[f].isin(top_k_vals), other='other').astype(str)
    df_test[bucketed_col]  = df_test[f].where(df_test[f].isin(top_k_vals),  other='other').astype(str)


bucket_cols = [f'{f}_bucket' for f in high_card_cols]
ohe_hc = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe_hc.fit(df_train[bucket_cols])
ohe_hc_col_names = ohe_hc.get_feature_names_out(bucket_cols).tolist()

df_train_ohe_hc = pd.DataFrame(ohe_hc.transform(df_train[bucket_cols]),
                                columns=ohe_hc_col_names, index=df_train.index, dtype=np.int8)
df_test_ohe_hc  = pd.DataFrame(ohe_hc.transform(df_test[bucket_cols]),
                                columns=ohe_hc_col_names, index=df_test.index,  dtype=np.int8)


df_train_ohe = pd.concat([df_train_ohe, df_train_ohe_hc], axis=1)
df_test_ohe  = pd.concat([df_test_ohe,  df_test_ohe_hc],  axis=1)

print(f'Итого OHE-признаков: {len(df_train_ohe.columns)} '
      f'(low-card: {len(ohe_col_names)}, high-card bucketed: {len(ohe_hc_col_names)})')

cat_feature_34: 120 значений → топ-1 покрывают 99.2% строк
cat_feature_39: 1989 значений → топ-159 покрывают 85.0% строк
cat_feature_52: 51 значений → топ-12 покрывают 85.8% строк
Итого OHE-признаков: 300 (low-card: 128, high-card bucketed: 172)


## 6. Финальная сборка трёх датасетов

- **tree_nan** — LightGBM, CatBoost: числовые с NaN, без `_isnan` флагов (модель сама обрабатывает)
- **tree_zero** — sklearn-деревья, XGBoost: NaN в 0, + `_isnan` флаги
- **linear** — логрег, SVM, : NaN в 0, + `_isnan` флаги, + масштабирование

In [12]:
base_new_cols   = whale_cols + ['count_not_null']
linear_new_cols = base_new_cols + interaction_cols

# 1. tree_nan: NaN остаются, без _isnan флагов 
tree_nan_train = pd.concat([
    df_train[[ID_COL]],
    df_train[num_cols + cat_cols + base_new_cols]
], axis=1)
tree_nan_test = pd.concat([
    df_test[[ID_COL]],
    df_test[num_cols + cat_cols + base_new_cols]
], axis=1)

#  2. tree_zero: NaN → 0, + _isnan флаги
tree_zero_train = pd.concat([
    df_train[[ID_COL]],
    _train_num,
    df_train[cat_cols + isnan_cols + base_new_cols]
], axis=1)
tree_zero_test = pd.concat([
    df_test[[ID_COL]],
    _test_num,
    df_test[cat_cols + isnan_cols + base_new_cols]
], axis=1)

# 3. linear: NaN в 0, масштабирование, энкодинг, _isnan, пары взаимодействия 
linear_train = pd.concat([
    df_train[[ID_COL]],
    df_train_qt,
    df_train_ohe,
    df_train[isnan_cols + linear_new_cols]
], axis=1)
linear_test = pd.concat([
    df_test[[ID_COL]],
    df_test_qt,
    df_test_ohe,
    df_test[isnan_cols + linear_new_cols]
], axis=1)

print('tree_nan: ', tree_nan_train.shape,  tree_nan_test.shape)
print('tree_zero:', tree_zero_train.shape, tree_zero_test.shape)
print('linear:   ', linear_train.shape,    linear_test.shape)

tree_nan:  (750000, 208) (250000, 208)
tree_zero: (750000, 340) (250000, 340)
linear:    (750000, 706) (250000, 706)


In [13]:
# Проверка пропусков в финальных датасетах
for name, df in [
    ('tree_nan_train',  tree_nan_train),
    ('tree_zero_train', tree_zero_train),
    ('linear_train',    linear_train),
]:
    n_null = df.drop(columns=[ID_COL]).isnull().sum().sum()
    print(f'{name}: пропусков = {n_null}')

tree_nan_train: пропусков = 50667394
tree_zero_train: пропусков = 0
linear_train: пропусков = 0


## 9. Сохранение

In [ ]:
tree_nan_train.to_parquet('tree_nan_train.parquet',   index=False)
tree_nan_test.to_parquet('tree_nan_test.parquet',     index=False)
tree_zero_train.to_parquet('tree_zero_train.parquet', index=False)
tree_zero_test.to_parquet('tree_zero_test.parquet',   index=False)
linear_train.to_parquet('linear_train.parquet',       index=False)
linear_test.to_parquet('linear_test.parquet',         index=False)

print('Сохранено:')
for name, df in [
    ('tree_nan_train ',  tree_nan_train),
    ('tree_nan_test  ',  tree_nan_test),
    ('tree_zero_train',  tree_zero_train),
    ('tree_zero_test ',  tree_zero_test),
    ('linear_train   ',  linear_train),
    ('linear_test    ',  linear_test),
]:
    print(f'  {name}: {df.shape}')

## Итог

| Датасет | Числовые | NaN | флаг nan | маштабирование | Файл |
|---|---|---|---|---|---|
| `tree_nan` | исходные |  остаются | X | X | `tree_nan_{train,test}.parquet` |
| `tree_zero` | исходные | 0 | Y | X | `tree_zero_{train,test}.parquet` |
| `linear` | масштабировали |  0 | Y | Y | `linear_{train,test}.parquet` |

Общие доп. признаки во всех трёх: `is_whale_`, `count_not_null`